<a href="https://colab.research.google.com/github/keivernunez/dataminingavanzado_austral/blob/main/Clase5/Note4_de_4_TabNet_FTTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparativo: TabNet vs FT-Transformer (rtdl_revisiting_models) vs Bosque Aleatorio

**Objetivo:** se comparan tres enfoques para regresión en datos tabulares (California Housing):
- TabNet — atención por columnas (máscara escasa secuencial)
- FT-Transformer (rtdl_revisiting_models) — transformer que aplica self-attention completa entre columnas
- Bosque Aleatorio — baseline clásico

También se visualizan máscaras de atención de TabNet y se procura extraer una aproximación de atención desde FT-Transformer (si la implementación lo permite).


## 0) Instalación (ejecutar si falta alguna dependencia)

En Jupyter local probablemente ya se tengan `scikit-learn`, `matplotlib`, `seaborn` y `torch`.
Si falta `pytorch-tabnet` o `rtdl_revisiting_models`, se ejecuta la celda siguiente (descomentar y ejecutar).


In [ ]:
%pip install pytorch-tabnet rtdl_revisiting_models --quiet
# Si hay problemas con versiones de torch, se instala una versión compatible de torch primero.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)


## 1) Cargar y preparar California Housing

In [ ]:
cal = fetch_california_housing(as_frame=True)
df = cal.frame
X = df.drop(columns='MedHouseVal').values
y = df['MedHouseVal'].values
feature_names = list(df.drop(columns='MedHouseVal').columns)

# División en entrenamiento/validación/prueba
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.3, random_state=seed)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=seed)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print('Forma del conjunto de entrenamiento:', X_train.shape)
print('Forma del conjunto de validación:', X_val.shape)
print('Forma del conjunto de prueba:', X_test.shape)


### Explicación de las características del conjunto de datos California Housing

El conjunto de datos California Housing contiene información sobre viviendas en distritos de California, derivado del censo de 1990. Las características son:
- **MedInc**: Ingreso mediano en el grupo de bloques (en decenas de miles de dólares).
- **HouseAge**: Edad mediana de las casas en el grupo de bloques.
- **AveRooms**: Promedio de habitaciones por hogar.
- **AveBedrms**: Promedio de dormitorios por hogar.
- **Population**: Población en el grupo de bloques.
- **AveOccup**: Promedio de ocupantes por hogar.
- **Latitude**: Latitud del grupo de bloques.
- **Longitude**: Longitud del grupo de bloques.

La variable objetivo es **MedHouseVal**: Valor mediano de las casas en cientos de miles de dólares.

## 2) Línea base: Bosque Aleatorio

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1)
start = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - start

def eval_model_preds(y_tr, y_v, y_te, preds_tr, preds_v, preds_te):
    return {
        'r2_tr': r2_score(y_tr, preds_tr),
        'r2_val': r2_score(y_v, preds_v),
        'r2_test': r2_score(y_te, preds_te),
        'mae_test': mean_absolute_error(y_te, preds_te),
        'rmse_test': root_mean_squared_error(y_te, preds_te)
    }

preds_tr = rf.predict(X_train)
preds_val = rf.predict(X_val)
preds_test = rf.predict(X_test)
rf_metrics = eval_model_preds(y_train, y_val, y_test, preds_tr, preds_val, preds_test)
rf_metrics['time'] = rf_time
rf_metrics


In [ ]:
imp_rf = rf.feature_importances_
plt.figure(figsize=(8,4))
sns.barplot(x=imp_rf, y=feature_names)
plt.title('Bosque Aleatorio - Importancia de características')
plt.tight_layout()
plt.show()


## 3) TabNet

Si no está instalado, se ejecuta `pip install pytorch-tabnet` antes.

In [ ]:
from pytorch_tabnet.tab_model import TabNetRegressor

clf = TabNetRegressor(seed=seed, verbose=0)
start = time.time()
clf.fit(
    X_train, y_train.reshape(-1, 1),
    eval_set=[(X_val, y_val.reshape(-1, 1))],
    max_epochs=100,
    patience=15,
    batch_size=256,
    virtual_batch_size=64
)
tabnet_time = time.time() - start

preds_tabnet = clf.predict(X_test).ravel()
metrics_tabnet = {
    'r2_test': r2_score(y_test, preds_tabnet),
    'mae_test': mean_absolute_error(y_test, preds_tabnet),
    'rmse_test': root_mean_squared_error(y_test, preds_tabnet),
    'time': tabnet_time
}

# Explicabilidad: TabNet puede devolver máscaras de atención
try:
    masks = clf.explain(X_test)
    mask_arr = np.array(masks[0])
    if mask_arr.ndim == 3:
        avg_mask = mask_arr.mean(axis=(0,1))
    elif mask_arr.ndim == 2:
        avg_mask = mask_arr.mean(axis=0)
    else:
        avg_mask = mask_arr.mean(axis=0)
except Exception as e:
    print('No se pudo extraer máscaras de TabNet:', e)
    avg_mask = None

metrics_tabnet, (avg_mask[:len(feature_names)] if avg_mask is not None else None)


In [ ]:
if avg_mask is not None:
    plt.figure(figsize=(8,4))
    sns.barplot(x=avg_mask, y=feature_names)
    plt.title('TabNet - Atención promedio por característica (máscara)')
    plt.tight_layout()
    plt.show()


## 4) FT-Transformer (rtdl_revisiting_models)

Si no está instalado, se ejecuta `pip install rtdl_revisiting_models`.
Se usa la clase FTTransformer de `rtdl_revisiting_models` para entrenar un FT-Transformer de manera sencilla.

In [ ]:
from rtdl_revisiting_models import FTTransformer

# Preparar tensores
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device).unsqueeze(1)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.float32).to(device).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

model = FTTransformer(
    n_cont_features=X_train.shape[1],
    cat_cardinalities=None,
    d_block=64,
    n_blocks=3,
    attention_n_heads=8,
    attention_dropout=0.1,
    ffn_d_hidden_multiplier=3.0,
    ffn_dropout=0.1,
    residual_dropout=0.0,
    d_out=1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

def train_epoch(model, X, y, batch_size=256):
    model.train()
    perm = np.random.permutation(X.shape[0])
    losses = []
    for i in range(0, X.shape[0], batch_size):
        idx = perm[i:i+batch_size]
        xb = X[idx]
        yb = y[idx]
        preds = model(xb, None)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)

start = time.time()
for epoch in range(30):
    tr_loss = train_epoch(model, X_train_t, y_train_t)
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t, None).cpu().numpy().squeeze()
    val_rmse = root_mean_squared_error(y_val, val_preds)
    if epoch % 5 == 0:
        print(f'Época {epoch:02d} pérdida_entrenamiento={tr_loss:.4f} rmse_validación={val_rmse:.4f}')
ft_time = time.time() - start

model.eval()
with torch.no_grad():
    preds_ft = model(X_test_t, None).cpu().numpy().squeeze()
metrics_ft = {
    'r2_test': r2_score(y_test, preds_ft),
    'mae_test': mean_absolute_error(y_test, preds_ft),
    'rmse_test': root_mean_squared_error(y_test, preds_ft),
    'time': ft_time
}
metrics_ft


### 4.1 Extraer atención desde FT-Transformer (se intenta con hooks)
Dependiendo de la implementación interna, puede que no sea posible obtener los pesos de atención sin modificar la librería. Se intentan registrar hooks sobre `nn.MultiheadAttention`.


In [ ]:
attn_weights_list = []
def save_attn_hook(module, input, output):
    try:
        if isinstance(output, tuple) and len(output) >= 2:
            attn = output[1]
        else:
            attn = None
    except Exception:
        attn = None
    if attn is not None:
        attn_weights_list.append(attn.detach().cpu().numpy())

hooks = []
for name, module in model.named_modules():
    if isinstance(module, nn.MultiheadAttention):
        hooks.append(module.register_forward_hook(save_attn_hook))

with torch.no_grad():
    _ = model(X_test_t[:128], None)

for h in hooks:
    h.remove()

if len(attn_weights_list) == 0:
    print('No se registraron pesos de atención (la implementación interna puede no exponerlos).')
    attn_map = None
else:
    arr = np.concatenate([a for a in attn_weights_list], axis=0)
    attn_map = arr.mean(axis=(0,1))
    print('Mapa de atención reconstruido con forma', attn_map.shape)

attn_map.shape if attn_map is not None else None


In [ ]:
if 'attn_map' in globals() and attn_map is not None:
    if attn_map.shape[0] == len(feature_names):
        importance_ft = attn_map.mean(axis=0)
        plt.figure(figsize=(8,4))
        sns.barplot(x=importance_ft, y=feature_names)
        plt.title('FT-Transformer - Importancia aproximada desde atención')
        plt.tight_layout()
        plt.show()
    else:
        print('Mapa de atención no coincide con número de características; no se puede mapear directamente')


## 5) Comparación final de métricas

In [ ]:
results = pd.DataFrame([
    {'model':'BosqueAleatorio', **rf_metrics},
    {'model':'TabNet', **metrics_tabnet},
    {'model':'FTTransformer', **metrics_ft}
])
results_display = results[['model','r2_test','mae_test','rmse_test','time']]
results_display


In [ ]:
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
sns.barplot(x='model', y='r2_test', data=results_display)
plt.title('R2 en prueba')
plt.subplot(1,2,2)
sns.barplot(x='model', y='rmse_test', data=results_display)
plt.title('RMSE en prueba')
plt.tight_layout()
plt.show()


## 6) Conclusiones y puntos docentes

- TabNet: máscaras interpretables; útil en conjuntos medianos; atención escasa por columnas.
- FT-Transformer: atención completa entre columnas; flexible para interacciones complejas; extraer atención puede requerir modificar implementación.
- Bosque Aleatorio: baseline sólido y rápido; feature_importances_ útil como referencia.

Se discute con los alumnos: ¿la atención mejora el rendimiento? ¿compensa la complejidad? ¿qué pasa si hay pocas muestras?
